<a href="https://colab.research.google.com/github/minjaaaa/Schneider-Electric-Internship/blob/main/notebooks/04_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!git clone -q https://github.com/minjaaaa/Schneider-Electric-Internship.git repo
%cd repo
!pip install -q qdrant-client FlagEmbedding sentence-transformers python-dotenv bitsandbytes accelerate

/content/repo/repo


In [13]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

Pretraga: konfiguracija eksperimenta 8 (hibrid, top 5)

In [14]:
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # generato - otvoren, bez prihvatanja licence
TOP_K = 5                                # broj chunk-ova u kontekstu (cover_rank: većina izvora je u top 5)
RUN_NAME = "gen_hybrid_qwen7b"

In [15]:
import sys, json
sys.path.insert(0, "src")
from chunking import load_jsonl
from config import DATA_PROCESSED, EVAL_QUESTIONS
from figures import figure_chunks
from indexing import get_client, ensure_index, hybrid_search

questions = json.loads(EVAL_QUESTIONS.read_text(encoding="utf-8"))
chunks = load_jsonl(DATA_PROCESSED / "chunks_section.jsonl") + figure_chunks()
client = get_client(in_memory=True)                     # Qdrant u memoriji; u Colab-u nema qdrant_data/
collection = ensure_index(client, chunks, with_path=True, hybrid=True)

retrieved = {q["id"]: hybrid_search(client, collection, q["question"], TOP_K, using="hybrid",
                                    exclude_superseded=True)
             for q in questions}
print(len(chunks), "chunk-ova;", len(retrieved), "pitanja pretraženo")

Indeksiram m3_283c9320e9 (108 chunk-ova)...


Inference Embeddings: 100%|██████████| 7/7 [00:06<00:00,  1.11it/s]


108 chunk-ova; 49 pitanja pretraženo


In [16]:
from generate import HFChatModel, answer_question
llm = HFChatModel(GEN_MODEL)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Probni odgovor na Q02 (memo menja pravilo)

In [17]:
q = next(q for q in questions if q["id"] == "Q02")      # memo menja pravilo: test pravila o statusu
r = answer_question(q["question"], retrieved["Q02"], llm)
print(r["raw"]); print(r["citations"], r["abstained"], r["latency_s"], "s")

{"answer": "160 EUR", "citations": ["PT-2026-02", "5.2"], "abstained": false}
['PT-2026-02', '5.2'] False 3.94 s


In [18]:
from datetime import datetime
from pathlib import Path
from tqdm.auto import tqdm

rows = []
for q in tqdm(questions):
    r = answer_question(q["question"], retrieved[q["id"]], llm)
    rows.append({"id": q["id"], "category": q["category"], "question": q["question"],
                 "expected_answer": q["expected_answer"], "sources": q["sources"], **r})

out = Path("results") / f"{datetime.now():%Y-%m-%d_%H%M}_{RUN_NAME}"
out.mkdir(parents=True, exist_ok=True)
with open(out / "answers.jsonl", "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
json.dump({"generator": GEN_MODEL, "top_k": TOP_K, "retriever": "8_hybrid (exclude_superseded, with_path, figures)"},
          open(out / "run_config.json", "w"), indent=2)
print("Sačuvano u", out, "| parse greške:", sum(r["parse_error"] for r in rows))

  0%|          | 0/49 [00:00<?, ?it/s]

Sačuvano u results/2026-10-14_1104_gen_hybrid_qwen7b | parse greške: 0


In [19]:
!zip -qr results_gen.zip {out}
from google.colab import files
files.download("results_gen.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>